<div dir="rtl">
<h1>یک راه مستقیم برای مقدار و مشتق</h1>
<p>درس 44 از 76 · چرا خروجی تبدیل را به ورودی اضافه می‌کنیم؟ · <code dir="ltr">38-residual</code></p>
<p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-03/38-residual.html">📖 بازگشت به همین درس</a></p>
<p>اثر مسیر جمع را در خروجی و Gradient جداگانه مشاهده کنید.</p><p>پیش‌نیاز: جمع عضو‌به‌عضو، requires_grad و backward را بشناسید.</p>
<p>این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو Cell با برچسب TODO را خودتان کامل کنید. پیام INCOMPLETE یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p>از بالا به پایین اجرا کنید. پس از تغییر هر تابع، Cell آن و سپس Cell آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code>Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl">
<h2>قبل از اجرا، پیش‌بینی کنید</h2>
<p>اگر زیرلایه اصلاح صفر بسازد، y=x+f(x) چه می‌شود؟ برای f(x)=2x مشتق چند است و برای f(x)=-x چه؟</p>
</div>

<div dir="rtl"><p>پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
x = torch.tensor([2.,-1.,3.],requires_grad=True)
print('input preserved by zero update:',x+torch.zeros_like(x))

<div dir="rtl">
<h2>این بار شما کد بنویسید</h2>
<p>تابع residual_update(x, scale) را برای f(x)=scale*x بنویسید. Tensor خروجی باید هم مقدار x را نگه دارد و هم مسیر مشتق مستقیم آن را؛ از detach یا ساخت Tensor تازه از عددها استفاده نکنید.</p>
</div>

In [ ]:
def residual_update(x, scale):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = residual_update(x,2.)
    if result is None: return False
    torch.testing.assert_close(result,3*x)
    grad = torch.autograd.grad(result.sum(),x)[0]
    torch.testing.assert_close(grad,torch.full_like(x,3.))
    for scale in (0.,-1.,0.5):
        z = torch.tensor([1.,4.],requires_grad=True)
        y = residual_update(z,scale)
        torch.testing.assert_close(y,(1+scale)*z)
        torch.testing.assert_close(torch.autograd.grad(y.sum(),z)[0],torch.full_like(z,1+scale))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>فقط یک عامل را تغییر دهید</h2>
<p>برای مشاهدهٔ مسیر مستقیم در بلوک واقعی، همهٔ Parameterهای آن را موقتاً صفر می‌کنیم؛ جزئیات داخل بلوک را در درس‌های بعد باز می‌کنیم. فقط گزینهٔ residual را تغییر دهید. ورودی و وزن‌ها ثابت‌اند. نتیجهٔ این حالت کنترل‌شده را با تضمین غیرصفرماندن Gradient در هر مدل اشتباه نگیرید.</p>
</div>

In [ ]:
from mini_gpt.config import ModelConfig
from mini_gpt.transformer import TransformerBlock
block = TransformerBlock(ModelConfig(8,4,4,1,1,0.)).eval()
with torch.no_grad():
    for parameter in block.parameters():
        parameter.zero_()
for enabled in (False,True):
    z = torch.arange(8.).reshape(1,2,4).requires_grad_()
    y = block(z,residual=enabled)
    gradient = torch.autograd.grad(y.sum(),z)[0]
    print('residual, output, input gradient:',enabled,y.detach(),gradient)

<div dir="rtl">
<h2>خرابی را پیدا کنید</h2>
<p>کد خراب x.detach را در میان‌بر می‌گذارد. مقدار خروجی همان است، ولی یک سهم مشتق حذف می‌شود. تابع intact_skip(x, update) را اصلاح کنید؛ update یک Tensor وابسته به x است.</p>
</div>

In [ ]:
z = torch.tensor(4.,requires_grad=True)
wrong = z.detach()+2*z
wrong.backward()
print('same output, missing skip gradient:',wrong.item(),z.grad.item())

<div dir="rtl">
<h2>اصلاح را خودتان بنویسید</h2>
<p>علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def intact_skip(x, update):
    # TODO
    return None

In [ ]:
def test_repair():
    z = torch.tensor([2.,3.],requires_grad=True)
    result = intact_skip(z,2*z)
    if result is None: return False
    torch.testing.assert_close(torch.autograd.grad(result.sum(),z)[0],torch.full_like(z,3.))
    z = torch.tensor(2.,requires_grad=True)
    assert torch.autograd.grad(intact_skip(z,0*z),z)[0].item() == 1.
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl">
<h2>در Mini-GPT کجا به کار می‌آید؟</h2>
<p>TransformerBlock دو بار جمع مستقیم دارد. در API واقعی، residual=False هر دو جمع را حذف می‌کند؛ این با detachکردن میان‌بر یا کم‌کردن x از خروجی نهایی یکسان نیست.</p>
</div>

<div dir="rtl">
<h2>با زبان خودتان توضیح دهید</h2>
<p>چرا آزمون فقط مقدار خروجی، خطای detach در مسیر مستقیم را پیدا نمی‌کند؟</p>
</div>
<div dir="rtl"><p>پیش‌بینی و مشاهدهٔ من: …</p><p>علت خرابی و اصلاح من: …</p></div>

<div dir="rtl"><p><a target="_self" href="http://127.0.0.1:8000/part-06/chapter-03/38-residual.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/38-residual.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>